# E4 NF Direction-Match Debug

Fast E4-only diagnostic for directly training the NF-induced preconditioner to match the explicit damped probe-metric direction.

This keeps the E4 task, train/probe/test split, target function, MLP width, parameter dimension, flow architecture, and downstream optimization checks fixed. Only the NF training objective changes from geometry-isometry to direction matching.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning.e4_debug import (
    E4DebugConfig,
    build_e4_debug_state,
)
from post_train_research.loss_landscape_analysis.flow_preconditioning.e4_direction_match import (
    direction_match_variant_slug,
    direction_target_summary,
    load_or_compute_direction_targets,
    run_scaled_probe_metric_curve,
    train_direction_match_flow,
)
from post_train_research.loss_landscape_analysis.flow_preconditioning.optimization import (
    run_direct_curves_batched,
    run_flow_curves_batched,
)

plt.rcParams['figure.dpi'] = 130


## Hardcoded Config

`rho=1e-1` is the main diagnostic. Add `1e-2` to `RHO_VALUES` for the optional second condition. Fast mode runs one `alpha` and three beta variants; set `RUN_FULL_ALPHA_TUNE=True` to run the full alpha grid.


In [ ]:
RUN_LABEL = 'e4_nf_direction_match_rq_spline_fast'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

RHO_VALUES = (1e-1,)
RUN_FULL_ALPHA_TUNE = False
DEFAULT_METRIC_ALPHA = 1e-3
FULL_METRIC_ALPHA_VALUES = (1e-6, 1e-4, 1e-3, 1e-2, 1e-1)
METRIC_ALPHA_VALUES = FULL_METRIC_ALPHA_VALUES if RUN_FULL_ALPHA_TUNE else (DEFAULT_METRIC_ALPHA,)
SCALE_BETAS = (0.0, 0.1, 1.0)
FORCE_TARGET_RECOMPUTE = False

BASE_DEBUG_KWARGS = dict(
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning/e4_direction_match_debug',
    device=DEVICE,
    dtype='float32',
    seed=0,
    flow_steps=80,
    eval_every=10,
    flow_batch_size=4,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_architecture='rq_spline',
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_spline_bins=8,
    flow_spline_bound=5.0,
    flow_random_samples=64,
    flow_trajectory_count=4,
    flow_trajectory_steps=8,
    heldout_geometry_samples=32,
    train_eval_samples=12,
    heldout_eval_samples=12,
)

BASE_DEBUG_KWARGS


## Build E4 States And Cached Targets

For each `(rho, alpha)` this precomputes and stores `g`, `G_P`, `(G_P + lambda I)^-1`, and `p_metric`. These targets are independent of NF parameters and are reused during NF training.


In [ ]:
states = {}
targets = {}
target_summary_rows = []

for rho in RHO_VALUES:
    debug_cfg = E4DebugConfig(
        run_label=f'{RUN_LABEL}_rho{rho:.0e}'.replace('-', 'm'),
        rho=float(rho),
        **BASE_DEBUG_KWARGS,
    )
    state = build_e4_debug_state(debug_cfg)
    states[float(rho)] = state
    print('rho:', rho, 'output_dir:', state.output_dir)
    print('flow_pool:', tuple(state.flow_pool.shape), 'heldout_pool:', tuple(state.heldout_pool.shape))

    for alpha in METRIC_ALPHA_VALUES:
        train_targets = load_or_compute_direction_targets(
            state=state,
            theta_samples=state.flow_pool,
            metric_alpha=float(alpha),
            split='train',
            force_recompute=FORCE_TARGET_RECOMPUTE,
        )
        heldout_targets = load_or_compute_direction_targets(
            state=state,
            theta_samples=state.heldout_pool,
            metric_alpha=float(alpha),
            split='heldout',
            force_recompute=FORCE_TARGET_RECOMPUTE,
        )
        targets[(float(rho), float(alpha), 'train')] = train_targets
        targets[(float(rho), float(alpha), 'heldout')] = heldout_targets
        print('  alpha:', alpha, 'train_targets:', tuple(train_targets.theta.shape), 'heldout_targets:', tuple(heldout_targets.theta.shape))
        target_summary_rows.extend([direction_target_summary(train_targets), direction_target_summary(heldout_targets)])

target_summary = pd.DataFrame(target_summary_rows)
display(target_summary)


## Train Direction-Match NF Variants

Loss is `1 - cos(p_NF, p_metric)` plus optional `beta * log_norm_ratio^2`.


In [ ]:
results = {}
history_tables = []
geometry_tables = []

for rho in RHO_VALUES:
    state = states[float(rho)]
    for alpha in METRIC_ALPHA_VALUES:
        train_targets = targets[(float(rho), float(alpha), 'train')]
        heldout_targets = targets[(float(rho), float(alpha), 'heldout')]
        for beta in SCALE_BETAS:
            slug = direction_match_variant_slug(rho=float(rho), metric_alpha=float(alpha), scale_beta=float(beta))
            print('training', slug)
            result = train_direction_match_flow(
                state=state,
                train_targets=train_targets,
                heldout_targets=heldout_targets,
                scale_beta=float(beta),
            )
            results[(float(rho), float(alpha), float(beta))] = result
            history_tables.append(result.history.assign(rho=float(rho), alpha=float(alpha), beta=float(beta), variant=slug))
            geometry_tables.append(result.final_geometry.assign(rho=float(rho), alpha=float(alpha), beta=float(beta), variant=slug))

all_history = pd.concat(history_tables, ignore_index=True)
all_geometry = pd.concat(geometry_tables, ignore_index=True)
summary = (
    all_history.sort_values('step')
    .groupby(['rho', 'alpha', 'beta', 'variant'], as_index=False)
    .tail(1)
    .sort_values(['direction_heldout_cos_median', 'direction_heldout_loss'], ascending=[False, True])
)

root = next(iter(states.values())).output_dir.parent
all_history.to_csv(root / 'all_direction_match_history.csv', index=False)
all_geometry.to_csv(root / 'all_direction_match_geometry.csv', index=False)
summary.to_csv(root / 'direction_match_summary.csv', index=False)
display(summary[['rho', 'alpha', 'beta', 'step', 'direction_train_cos_median', 'direction_heldout_cos_median', 'direction_heldout_norm_ratio_metric_to_nf_median', 'heldout_flow_R', 'heldout_R_ratio', 'variant']])
print('saved root:', root)


## Loss Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

plot_rows = all_history[all_history['step'] > 0]
for variant, rows in plot_rows.groupby('variant'):
    axes[0, 0].plot(rows['step'], rows['batch_loss'], label=variant)
    axes[0, 1].plot(rows['step'], rows['batch_dir_loss'], label=variant)
    axes[1, 0].plot(rows['step'], rows['batch_scale_loss'], label=variant)
    axes[1, 1].plot(rows['step'], rows['direction_heldout_loss'], label=variant)

axes[0, 0].set_title('train batch total loss')
axes[0, 1].set_title('train batch direction loss')
axes[1, 0].set_title('train batch scale loss')
axes[1, 1].set_title('heldout direction-match loss')
for ax in axes.ravel():
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.25)
axes[0, 0].legend(fontsize=7)

loss_figure_path = root / 'direction_match_loss_curves.png'
fig.savefig(loss_figure_path, bbox_inches='tight')
print('saved:', loss_figure_path)


## Direction-Match And Geometry Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

for variant, rows in all_history.groupby('variant'):
    axes[0, 0].plot(rows['step'], rows['direction_heldout_cos_median'], label=variant)
    axes[0, 1].plot(rows['step'], rows['direction_heldout_loss'], label=variant)
    axes[1, 0].plot(rows['step'], rows['direction_heldout_norm_ratio_metric_to_nf_median'], label=variant)
    axes[1, 1].plot(rows['step'], rows['heldout_flow_R'], label=variant)

axes[0, 0].set_title('heldout cos(p_NF, p_metric)')
axes[0, 1].set_title('heldout direction objective')
axes[1, 0].set_title('heldout ||p_metric|| / ||p_NF||')
axes[1, 1].set_title('heldout pullback R')
for ax in axes.ravel():
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.25)
axes[0, 0].legend(fontsize=7, ncols=1)

figure_path = root / 'direction_match_training_curves.png'
fig.savefig(figure_path, bbox_inches='tight')
print('saved:', figure_path)


## Pick Best Variant

Best variant is selected by final held-out median cosine, with lower direction loss as tie-breaker. This is only for plotting and the small downstream check below.


In [ ]:
best_row = summary.iloc[0]
best_key = (float(best_row['rho']), float(best_row['alpha']), float(best_row['beta']))
best_result = results[best_key]
best_state = states[float(best_row['rho'])]

display(best_row.to_frame('best').T)
print('best output_dir:', best_result.output_dir)


## Small Raw vs NF vs Scaled Probe-Metric Optimization Check

This mirrors the existing E4 debug notebook, but the explicit probe-metric baseline uses the same scale-normalized damping `lambda = alpha * tr(G_P)/d + eps` as the direction targets.


In [ ]:
OPT_COMPARE_STARTS = 8
OPT_COMPARE_STEPS = 120
OPT_COMPARE_RAW_LR = 1e-3
OPT_COMPARE_NF_LR = 1e-3
OPT_COMPARE_PROBE_LR = 1e-2

starts = best_state.problem.sample_starts(OPT_COMPARE_STARTS, seed=best_state.debug_cfg.seed + 1234)
lrs_raw = torch.full((OPT_COMPARE_STARTS,), OPT_COMPARE_RAW_LR, device=starts.device, dtype=starts.dtype)
lrs_nf = torch.full((OPT_COMPARE_STARTS,), OPT_COMPARE_NF_LR, device=starts.device, dtype=starts.dtype)

raw_curves = run_direct_curves_batched(
    train_loss_fn=best_state.problem.train_loss,
    test_loss_fn=best_state.problem.test_loss,
    theta0_batch=starts,
    optimizer_name='adam',
    lrs=lrs_raw,
    steps=OPT_COMPARE_STEPS,
)
nf_curves = run_flow_curves_batched(
    train_loss_fn=best_state.problem.train_loss,
    test_loss_fn=best_state.problem.test_loss,
    flow=best_result.flow,
    theta0_batch=starts,
    optimizer_name='adam',
    lrs=lrs_nf,
    steps=OPT_COMPARE_STEPS,
)
probe_metric_curves = [
    run_scaled_probe_metric_curve(
        train_loss_fn=best_state.problem.train_loss,
        test_loss_fn=best_state.problem.test_loss,
        probe=best_state.probe,
        theta0=starts[idx],
        lr=OPT_COMPARE_PROBE_LR,
        steps=OPT_COMPARE_STEPS,
        metric_alpha=float(best_row['alpha']),
    )
    for idx in range(OPT_COMPARE_STARTS)
]

curve_rows = []
for method, curves, lr_value in [
    ('raw_adam', raw_curves, OPT_COMPARE_RAW_LR),
    ('nf_adam', nf_curves, OPT_COMPARE_NF_LR),
    ('scaled_probe_metric', probe_metric_curves, OPT_COMPARE_PROBE_LR),
]:
    for start_idx, curve in enumerate(curves):
        for step, (train_loss, test_loss) in enumerate(zip(curve.train_loss, curve.test_loss, strict=True)):
            curve_rows.append(dict(method=method, start_index=start_idx, step=step, train_loss=float(train_loss), test_loss=float(test_loss), lr=float(lr_value)))

optimization_curves = pd.DataFrame(curve_rows)
final_step_rows = optimization_curves[optimization_curves['step'] == OPT_COMPARE_STEPS]
best_rows = optimization_curves.groupby('method', as_index=False).agg(best_train_loss=('train_loss', 'min'))
optimization_summary = final_step_rows.groupby('method', as_index=False).agg(
    final_train_loss=('train_loss', 'median'),
    final_test_loss=('test_loss', 'median'),
).merge(best_rows, on='method', how='left')
optimization_curves.to_csv(best_result.output_dir / 'small_raw_nf_scaled_probe_metric_optimization_curves.csv', index=False)
optimization_summary.to_csv(best_result.output_dir / 'small_raw_nf_scaled_probe_metric_optimization_summary.csv', index=False)
display(optimization_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, loss_col, title in [(axes[0], 'train_loss', 'train loss'), (axes[1], 'test_loss', 'test loss')]:
    for method, rows in optimization_curves.groupby('method'):
        pivot = rows.pivot(index='step', columns='start_index', values=loss_col)
        median = pivot.median(axis=1)
        q25 = pivot.quantile(0.25, axis=1)
        q75 = pivot.quantile(0.75, axis=1)
        ax.plot(median.index, median.values, label=method)
        ax.fill_between(median.index, q25.values, q75.values, alpha=0.12)
    ax.set_yscale('log')
    ax.set_xlabel('step')
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.25)
    ax.legend()

opt_figure_path = best_result.output_dir / 'small_raw_nf_scaled_probe_metric_optimization.png'
fig.savefig(opt_figure_path, bbox_inches='tight')
print('saved:', opt_figure_path)


## Generated Interpretation


In [ ]:
best_final = best_result.history.iloc[-1]
raw_final = optimization_summary.set_index('method').loc['raw_adam']
nf_final = optimization_summary.set_index('method').loc['nf_adam']
probe_final = optimization_summary.set_index('method').loc['scaled_probe_metric']

interpretation = f'''
### Direction-Match Diagnostic Summary

- Best variant: `{best_row['variant']}`.
- Held-out median `cos(p_NF, p_metric)` ended at `{float(best_final['direction_heldout_cos_median']):.4f}`.
- Held-out norm ratio `||p_metric|| / ||p_NF||` ended at `{float(best_final['direction_heldout_norm_ratio_metric_to_nf_median']):.4g}`.
- Held-out geometry R changed from `{float(best_final['heldout_original_R']):.4g}` to `{float(best_final['heldout_flow_R']):.4g}`.
- Small optimization final train loss: raw Adam `{float(raw_final['final_train_loss']):.4g}`, NF Adam `{float(nf_final['final_train_loss']):.4g}`, scaled probe metric `{float(probe_final['final_train_loss']):.4g}`.

Interpretation rule of thumb: if direction cosine improves but NF optimization still does not improve, the remaining issue is likely scale/step-size or optimizer LR. If cosine does not improve, the flow architecture/objective is not learning the explicit metric direction on this target pool.
'''
display(Markdown(interpretation))
(best_result.output_dir / 'direction_match_interpretation.md').write_text(interpretation, encoding='utf-8')


## Files Written


In [ ]:
for path in sorted(root.glob('**/*')):
    if path.is_file():
        print(path)
